In this practical assignment, you will forecast hourly air pollution in Paris for the next three weeks. You will compare simple baselines with a zero-shot time-series foundation model, then explain what works, what does not, and why.

### Part One: Data understanding and preparation

In [36]:
import pandas as pd

df_train = pd.read_csv('../data/train.csv')

df_test = pd.read_csv('../data/test.csv')

 Identify the timestamp column and the five pollutant columns.
Check that dates are sorted and that there are no unexpected duplicates.

In [37]:
df_train.head()

,id,valeur_NO2,valeur_CO,valeur_O3,valeur_PM10,valeur_PM25
0,2020-01-01 00,42.9,0.718,15.7,73.1,64.4
1,2020-01-01 01,33.6,0.587,10.1,74.8,66.0
2,2020-01-01 02,29.3,NaN,5.1,51.0,44.9
3,2020-01-01 03,30.5,0.246,7.2,27.7,25.1
4,2020-01-01 04,29.3,0.204,8.3,15.3,13.6


In [38]:
df_train['id'] = pd.to_datetime(df_train['id'])

In [39]:
df_train = df_train.sort_values('id', ascending=True)

In [40]:
duplicates = df_train[df_train.duplicated()]
print(duplicates)

Empty DataFrame
Columns: [id, valeur_NO2, valeur_CO, valeur_O3, valeur_PM10, valeur_PM25]
Index: []


Check data quality: time range, missing values, duplicates, outliers and simple statistics for each
pollutant.

In [41]:
pollutants = ['valeur_NO2', 'valeur_CO', 'valeur_O3', 'valeur_PM10', 'valeur_PM25']

for pollutant in pollutants:
    print(f"Pollutant: {pollutant}")
    print("========================")
    print(f"Time range: {df_train[~df_train[pollutant].isna()]['id'].min()} - {df_train[~df_train[pollutant].isna()]['id'].max()}")
    print(f"Number of missing values: {df_train[pollutant].isna().sum()}")
    print(f"Range of values: {df_train[pollutant].min()} - {df_train[pollutant].max()}")
    print(f"Simple stats:\nMedian: {df_train[pollutant].median():.2f}\nMean: {df_train[pollutant].mean():.2f}\nStandard Deviation: {df_train[pollutant].std():.2f}\n")

Pollutant: valeur_NO2
Time range: 2020-01-01 00:00:00 - 2024-09-01 01:00:00
Number of missing values: 3297
Range of values: 1.1 - 131.0
Simple stats:
Median: 17.60
Mean: 21.83
Standard Deviation: 14.66

Pollutant: valeur_CO
Time range: 2020-01-01 00:00:00 - 2024-09-03 22:00:00
Number of missing values: 12529
Range of values: 0.037 - 4.309
Simple stats:
Median: 0.18
Mean: 0.20
Standard Deviation: 0.10

Pollutant: valeur_O3
Time range: 2020-01-01 00:00:00 - 2024-09-03 22:00:00
Number of missing values: 693
Range of values: -1.9 - 193.1
Simple stats:
Median: 50.70
Mean: 50.57
Standard Deviation: 26.49

Pollutant: valeur_PM10
Time range: 2020-01-01 00:00:00 - 2024-09-03 22:00:00
Number of missing values: 7167
Range of values: 0.5 - 128.5
Simple stats:
Median: 15.60
Mean: 18.22
Standard Deviation: 11.28

Pollutant: valeur_PM25
Time range: 2020-01-01 00:00:00 - 2024-09-03 22:00:00
Number of missing values: 1791
Range of values: 0.0 - 111.1
Simple stats:
Median: 8.70
Mean: 11.05
Standard Devi

In [42]:
def gap_lengths(s):
    is_na = s.isna()
    run_id = (is_na != is_na.shift()).cumsum()
    return is_na.groupby(run_id).sum()[is_na.groupby(run_id).first()]

for col in pollutants:
    g = gap_lengths(df_train[col])
    print(col)
    print("==========")
    print(f"Gaps: {len(g)}")
    print(f"Median: {g.median()}")
    print(f"Max: {g.max()}")
    print(f"Share of mising values in gaps larger than 6 hours: {g[g > 6].sum() / g.sum():.2f}\n")

valeur_NO2
Gaps: 161
Median: 1.0
Max: 1500
Share of mising values in gaps larger than 6 hours: 0.93

valeur_CO
Gaps: 411
Median: 1.0
Max: 10949
Share of mising values in gaps larger than 6 hours: 0.96

valeur_O3
Gaps: 71
Median: 3.0
Max: 131
Share of mising values in gaps larger than 6 hours: 0.81

valeur_PM10
Gaps: 749
Median: 8.0
Max: 90
Share of mising values in gaps larger than 6 hours: 0.90

valeur_PM25
Gaps: 209
Median: 2.0
Max: 493
Share of mising values in gaps larger than 6 hours: 0.81



In [44]:
df = df_train.copy()
df["timestamp"] = pd.to_datetime(df["id"], format="%Y-%m-%d")
df = df.set_index("id").sort_index()

def gap_table(s):
    is_na = s.isna()
    run_id = (is_na != is_na.shift()).cumsum()
    g = s.index.to_series().groupby(run_id).agg(['first', 'last', 'size'])
    return g[is_na.groupby(run_id).first()].sort_values('size', ascending=False)

for col in pollutants:
    print(col); print(gap_table(df[col]).head(5))

valeur_NO2
                         first                last  size
valeur_NO2                                              
156        2021-01-01 00:00:00 2021-03-04 11:00:00  1500
126        2020-09-03 14:00:00 2020-09-29 13:00:00   624
310        2024-07-31 15:00:00 2024-08-14 09:00:00   331
166        2021-04-11 02:00:00 2021-04-19 12:00:00   203
322        2024-09-01 02:00:00 2024-09-03 22:00:00    69
valeur_CO
                        first                last   size
valeur_CO                                               
748       2021-01-05 09:00:00 2022-04-06 13:00:00  10949
750       2022-04-27 08:00:00 2022-05-11 12:00:00    341
764       2022-09-15 17:00:00 2022-09-26 10:00:00    258
790       2023-08-11 18:00:00 2023-08-16 11:00:00    114
158       2020-03-12 09:00:00 2020-03-16 14:00:00    102
valeur_O3
                        first                last  size
valeur_O3                                              
18        2020-06-02 19:00:00 2020-06-08 05:00:00   131
8  

In [51]:
# check windows which have a low pct of missing values across all pollutants. Ideally in September to match the test dataset

window = pd.Timedelta(days=21)
rows = []
for start in pd.date_range("2022-06-01", "2024-07-01", freq="7D"):
    seg = df.loc[start:start+window-pd.Timedelta(hours=1), pollutants]
    rows.append({"start": start, **seg.isna().mean().round(3).to_dict()})
miss = pd.DataFrame(rows).set_index("start")
miss[miss.max(axis=1) < 0.05]   # windows with <5% missing in every pollutant

,valeur_NO2,valeur_CO,valeur_O3,valeur_PM10,valeur_PM25
start,,,,,
2022-11-02,0.002,0.000,0.010,0.006,0.000
2022-11-09,0.002,0.000,0.010,0.000,0.000
2022-11-16,0.000,0.000,0.000,0.002,0.002
2022-11-23,0.002,0.000,0.000,0.004,0.004
2022-11-30,0.002,0.002,0.000,0.006,0.006
2022-12-07,0.002,0.002,0.000,0.004,0.004
2022-12-14,0.000,0.004,0.000,0.002,0.002
2022-12-21,0.008,0.010,0.008,0.004,0.004
2022-12-28,0.008,0.026,0.008,0.004,0.004


Plot the pollutant series and describe visible trends, daily or weekly patterns, peaks and relationships between pollutants.

In [8]:
import plotly.express as px

fig = px.line(df_train, x='id', y=df_train.columns[1:6], title='Paris Air Quality Pollutants', labels={"id": "Date", "value": "μg/m³"})
fig.show()

In [49]:
# Zoom in one weekly trends:

mask = (df_train['id'] >= pd.to_datetime('2023-12-01')) & (df_train['id'] <= pd.to_datetime('2023-12-07'))

fig = px.line(df_train[mask], x='id', y=df_train.columns[1:6], title='Paris Air Quality Pollutants, One Week View', labels={"id": "Date", "value": "μg/m³"})
fig.show()

In [26]:
# Zoom in on daily trends:

mask = df_train['id'].dt.date == pd.Timestamp('2023-01-02').date()

fig = px.line(df_train[mask], x='id', y=df_train.columns[1:6], title='Paris Air Quality Pollutants, One Day View', labels={"id": "Date", "value": "μg/m³"})
fig.show()

### Part Two: Validation Strategy and Baselines

Before using a foundation model, build simple baselines. They are useful references and are often hard to beat on seasonal data.

Create a time-based validation split with a 504-hour forecast horizon. Make sure no future information enters the training data.

In [55]:
import numpy as np

df_train = df_train.set_index("id").sort_index()

HOURS = 504
val_start = pd.Timestamp("2023-10-18 00:00")
val_end  = val_start + pd.Timedelta(hours=HOURS - 1)   # 2023-11-07 23:00

# 1. value-level cleaning (no future info involved)
df_train.loc[df_train["valeur_O3"] < 0, "valeur_O3"] = np.nan

# 2. split: train is strictly before validation
train = df_train.loc[: val_start - pd.Timedelta(hours=1)].copy()
val  = df_train.loc[val_start : val_end].copy()

In [56]:
def gap_len(s):
    """For each NaN, the length of the gap it belongs to (0 if observed)."""
    is_na = s.isna()
    grp = (is_na != is_na.shift()).cumsum()
    return is_na.groupby(grp).transform("sum").where(is_na, 0)

def impute_past_only(s, profile, short=6, mid=48):
    gap = gap_len(s)
    out = s.copy()

    # <= 6 h: forward fill (uses only the last observed value)
    ff = s.ffill(limit=short)
    m_short = gap.between(1, short)
    out[m_short] = ff[m_short]

    # 7-48 h: weekday x hour profile from TRAIN only
    key = pd.MultiIndex.from_arrays([s.index.dayofweek, s.index.hour])
    prof = pd.Series(profile.reindex(key).values, index=s.index)
    m_mid = (gap > short) & (gap <= mid)
    out[m_mid] = prof[m_mid]

    # > 48 h: stays NaN
    return out, (m_short | m_mid)

train_imp = train.copy()
for col in pollutants:
    prof = train[col].groupby([train.index.dayofweek, train.index.hour]).mean()
    train_imp[col], flag = impute_past_only(train[col], prof)
    train_imp[col + "_imputed"] = flag

In [57]:
train_imp[pollutants].isna().mean()

valeur_NO2     0.071729
valeur_CO      0.356031
valeur_O3      0.011394
valeur_PM10    0.015031
valeur_PM25    0.030093
dtype: float64

Implement at least two baselines: persistence and seasonal naive. For the seasonal naive model, test a 168-hour lag and justify it.

In [16]:
# persistence: repeat the last observed value of each pollutant over the whole horizon

def persistence(train, index):
    last = train[pollutants].ffill().iloc[-1]          # last non-missing value per pollutant
    return pd.DataFrame({col: np.full(len(index), last[col]) for col in pollutants}, index=index)

pred_persist = persistence(train_imp, val.index)

In [17]:
# seasonal naive: repeat the last L observed hours, tiled over the horizon

def seasonal_naive(train, index, L=168):
    hist = train[pollutants]
    cycle = hist.iloc[-L:].copy()
    # if a value is still missing in the last cycle, fall back to the most recent value observed at the same position in the cycle
    for k in range(2, 6):
        cycle = cycle.fillna(pd.DataFrame(hist.iloc[-k * L:-(k - 1) * L].values, index=cycle.index, columns=pollutants))
    cycle = cycle.fillna(hist.ffill().iloc[-1])        # last resort: persistence
    reps = int(np.ceil(len(index) / L))
    pred = np.tile(cycle.values, (reps, 1))[:len(index)]
    return pd.DataFrame(pred, index=index, columns=pollutants)

pred_snaive = seasonal_naive(train_imp, val.index, L=168)

In [18]:
# evaluation: MAE per pollutant and average, on the observed validation values only (NaNs are skipped)

def mae_per_pollutant(y_true, y_pred):
    mae = (y_true[pollutants] - y_pred[pollutants]).abs().mean()   # mean() skips NaNs
    mae["average"] = mae[pollutants].mean()
    return mae

# test several lags for the seasonal naive: 24h (daily), 168h (weekly), 336h (two weeks)
candidates = {"persistence": pred_persist}
for L in [24, 168, 336]:
    candidates[f"seasonal naive (L={L})"] = seasonal_naive(train_imp, val.index, L=L)

results = pd.DataFrame({name: mae_per_pollutant(val, pred) for name, pred in candidates.items()}).T
results.round(3)

,valeur_NO2,valeur_CO,valeur_O3,valeur_PM10,valeur_PM25,average
persistence,7.915,0.043,18.598,5.023,2.940,6.904
seasonal naive (L=24),9.756,0.073,20.258,8.590,3.275,8.390
seasonal naive (L=168),10.985,0.045,17.535,9.491,5.073,8.626
seasonal naive (L=336),28.361,0.177,25.282,12.288,8.676,14.957


In [19]:
# observed vs baselines on the validation period

from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=len(pollutants), cols=1, shared_xaxes=True, subplot_titles=pollutants)
for i, col in enumerate(pollutants, start=1):
    fig.add_trace(go.Scatter(x=val.index, y=val[col], name="observed", line=dict(color="black"), legendgroup="observed", showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=val.index, y=pred_persist[col], name="persistence", line=dict(color="red"), legendgroup="persistence", showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=val.index, y=pred_snaive[col], name="seasonal naive (L=168)", line=dict(color="blue"), legendgroup="snaive", showlegend=(i == 1)), row=i, col=1)
fig.update_layout(title="Baselines vs observed, validation period", height=1100)
fig.show()

### Part Three: Zero-shot forecasting experiment

Run the foundation model in zero-shot mode. Record the model name, version, hardware and main settings.

In [63]:
# On Google Colab, uncomment and run once (then restart the runtime if asked):
# !pip install -q "chronos-forecasting>=2.1" torch

import platform
import importlib.metadata as importlib_metadata
import torch
from chronos import Chronos2Pipeline

MODEL_ID = "amazon/chronos-2"     # pretrained, used as is: no fine-tuning on the Paris data
CONTEXT = 8192                    # hours of history given to the model (about 11 months)
QUANTILE = 0.5                    # median forecast, the optimal point forecast for MAE

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
hardware = torch.cuda.get_device_name(0) if device == "cuda" else f"{platform.processor() or platform.machine()} ({device})"
pipe = Chronos2Pipeline.from_pretrained(MODEL_ID, device_map=device)

print(f"Model: {MODEL_ID} (120M parameters)")
print(f"chronos-forecasting {importlib_metadata.version('chronos-forecasting')}, torch {torch.__version__}")
print(f"Hardware: {hardware}")
print(f"Settings: context={CONTEXT}h, horizon={HOURS}h, point forecast=quantile {QUANTILE}, no covariates, no fine-tuning")

/Users/oliviayoung/Documents/HEC/time_series/paris-air-quality-forecasting/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1018 21:53:45.128000 90349 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 170/170 [00:00<00:00, 11990.38it/s]


Model: amazon/chronos-2 (120M parameters)
chronos-forecasting 2.3.2, torch 2.14.1
Hardware: arm (mps)
Settings: context=8192h, horizon=504h, point forecast=quantile 0.5, no covariates, no fine-tuning


Prepare the data in the format expected by the model (long format: one row per timestamp and series). Only the imputed training history (everything before the validation period) is passed as context.

In [64]:
def chronos_forecast(history, horizon_index, joint=True, context=CONTEXT):
    """Zero-shot Chronos-2 median forecast for all pollutants.

    joint=True : one series with 5 targets, so the model can use the relations between pollutants
    joint=False: five separate univariate series, forecast independently of each other
    """
    h = history[pollutants].iloc[-context:].rename_axis("timestamp").reset_index()
    if joint:
        ctx = h.assign(item_id="paris")
        target = pollutants
    else:
        ctx = h.melt(id_vars="timestamp", var_name="item_id", value_name="value")
        target = ["value"]

    out = pipe.predict_df(ctx, prediction_length=len(horizon_index), quantile_levels=[QUANTILE],
                          id_column="item_id", timestamp_column="timestamp", target=target)
    if joint:
        pred = out.pivot(index="timestamp", columns="target_name", values=str(QUANTILE))
    else:
        pred = out.pivot(index="timestamp", columns="item_id", values=str(QUANTILE))
    pred = pred[pollutants]
    assert (pred.index == horizon_index).all(), "forecast timestamps do not match the horizon"
    return pred

Generate forecasts for the full 504-hour horizon, align them with the validation timestamps and compute the MAE.

In [65]:
pred_chronos_joint = chronos_forecast(train_imp, val.index, joint=True)

results_fm = pd.DataFrame({
    "persistence": mae_per_pollutant(val, pred_persist),
    "seasonal naive (L=168)": mae_per_pollutant(val, pred_snaive),
    "Chronos-2 joint": mae_per_pollutant(val, pred_chronos_joint),
}).T
results_fm.round(3)

,valeur_NO2,valeur_CO,valeur_O3,valeur_PM10,valeur_PM25,average
persistence,7.915,0.043,18.598,5.023,2.940,6.904
seasonal naive (L=168),10.985,0.045,17.535,9.491,5.073,8.626
Chronos-2 joint,14.773,0.081,19.399,9.390,5.616,9.852


### Part Four: Comparative experiments

**Independent vs joint forecasting.** Same Chronos-2 weights, same context and same validation period. The only change is that each pollutant is forecast on its own.

In [66]:
pred_chronos_indep = chronos_forecast(train_imp, val.index, joint=False)

pd.DataFrame({
    "Chronos-2 independent": mae_per_pollutant(val, pred_chronos_indep),
    "Chronos-2 joint": mae_per_pollutant(val, pred_chronos_joint),
}).T.round(3)

,valeur_NO2,valeur_CO,valeur_O3,valeur_PM10,valeur_PM25,average
Chronos-2 independent,19.234,0.076,20.171,9.087,4.081,10.530
Chronos-2 joint,14.773,0.081,19.399,9.390,5.616,9.852


**Foundation model vs neural network.** A simple LSTM trained only on the Paris training data. It reads the last 336 hours (the 5 standardised pollutants plus hour and weekday encodings) and outputs the 504 x 5 forecast directly. It is trained with an L1 loss (the MAE metric) on sliding windows taken strictly inside the training period, with fixed hyperparameters (nothing is tuned on the validation period).

In [67]:
import torch.nn as nn

LOOKBACK, LSTM_EPOCHS, LSTM_HIDDEN, STRIDE = 336, 25, 64, 6

def calendar_features(index):
    hour, dow = 2 * np.pi * index.hour / 24, 2 * np.pi * index.dayofweek / 7
    return np.column_stack([np.sin(hour), np.cos(hour), np.sin(dow), np.cos(dow)])

class LSTMForecaster(nn.Module):
    def __init__(self, n_in, n_out, horizon, hidden=LSTM_HIDDEN):
        super().__init__()
        self.horizon, self.n_out = horizon, n_out
        self.lstm = nn.LSTM(n_in, hidden, batch_first=True)
        self.head = nn.Linear(hidden, horizon * n_out)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.head(h[-1]).view(-1, self.horizon, self.n_out)

def make_inputs(values_std, index):
    # missing values (long gaps left by the imputation) are set to the mean (0 after standardisation)
    return np.hstack([np.nan_to_num(values_std), calendar_features(index)]).astype("float32")

def fit_lstm(history, horizon=HOURS, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    vals = history[pollutants].values
    mu, sd = np.nanmean(vals, axis=0), np.nanstd(vals, axis=0)
    std = (vals - mu) / sd
    X_all = make_inputs(std, history.index)

    starts = [s for s in range(0, len(history) - LOOKBACK - horizon + 1, STRIDE)
              if np.isfinite(std[s + LOOKBACK : s + LOOKBACK + horizon]).mean() > 0.5]
    X = torch.tensor(np.stack([X_all[s : s + LOOKBACK] for s in starts]))
    Y = torch.tensor(np.stack([std[s + LOOKBACK : s + LOOKBACK + horizon] for s in starts]), dtype=torch.float32)

    model = LSTMForecaster(X.shape[-1], len(pollutants), horizon)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(LSTM_EPOCHS):
        model.train(); perm = torch.randperm(len(X)); total = 0
        for i in range(0, len(X), 64):
            idx = perm[i : i + 64]
            pred, y = model(X[idx]), Y[idx]
            mask = ~torch.isnan(y)
            loss = (pred - torch.nan_to_num(y)).abs()[mask].mean()      # L1 loss on observed targets only
            opt.zero_grad(); loss.backward(); opt.step(); total += loss.item() * len(idx)
        if epoch % 5 == 4: print(f"epoch {epoch + 1}: train L1 (standardised) = {total / len(X):.4f}")
    return model, mu, sd

def forecast_lstm(fitted, history, horizon_index):
    model, mu, sd = fitted
    std = (history[pollutants].values[-LOOKBACK:] - mu) / sd
    x = torch.tensor(make_inputs(std, history.index[-LOOKBACK:]))[None]
    model.eval()
    with torch.no_grad():
        out = model(x)[0].numpy() * sd + mu
    return pd.DataFrame(np.clip(out, 0, None), index=horizon_index, columns=pollutants)

lstm_fit = fit_lstm(train_imp)
pred_lstm = forecast_lstm(lstm_fit, train_imp, val.index)

epoch 5: train L1 (standardised) = 0.5963
epoch 10: train L1 (standardised) = 0.5825
epoch 15: train L1 (standardised) = 0.5790
epoch 20: train L1 (standardised) = 0.5937
epoch 25: train L1 (standardised) = 0.5806


In [68]:
results_cmp = pd.DataFrame({
    "Chronos-2 independent": mae_per_pollutant(val, pred_chronos_indep),
    "Chronos-2 joint": mae_per_pollutant(val, pred_chronos_joint),
    "LSTM": mae_per_pollutant(val, pred_lstm),
}).T
results_cmp.round(3)

,valeur_NO2,valeur_CO,valeur_O3,valeur_PM10,valeur_PM25,average
Chronos-2 independent,19.234,0.076,20.171,9.087,4.081,10.530
Chronos-2 joint,14.773,0.081,19.399,9.390,5.616,9.852
LSTM,8.888,0.033,14.978,5.949,3.641,6.698


### Part Five: Results and discussion

Compare all models using MAE per pollutant and average MAE.

In [69]:
preds = {
    "persistence": pred_persist,
    "seasonal naive (L=168)": pred_snaive,
    "Chronos-2 independent": pred_chronos_indep,
    "Chronos-2 joint": pred_chronos_joint,
    "LSTM": pred_lstm,
}
results_all = pd.DataFrame({name: mae_per_pollutant(val, pred) for name, pred in preds.items()}).T
display(results_all.round(3))

# The average MAE is in raw units, so it is dominated by the large-scale pollutants (O3, NO2).
# To see where the average hides differences, compare each model to seasonal naive, pollutant by pollutant (1 = same, <1 = better).
display((results_all[pollutants] / results_all.loc["seasonal naive (L=168)", pollutants]).round(2))
print("Best model per pollutant:\n", results_all[pollutants].idxmin().to_string())
print("Best model on the average MAE:", results_all["average"].idxmin())

,valeur_NO2,valeur_CO,valeur_O3,valeur_PM10,valeur_PM25,average
persistence,7.915,0.043,18.598,5.023,2.940,6.904
seasonal naive (L=168),10.985,0.045,17.535,9.491,5.073,8.626
Chronos-2 independent,19.234,0.076,20.171,9.087,4.081,10.530
Chronos-2 joint,14.773,0.081,19.399,9.390,5.616,9.852
LSTM,8.888,0.033,14.978,5.949,3.641,6.698


,valeur_NO2,valeur_CO,valeur_O3,valeur_PM10,valeur_PM25
persistence,0.72,0.95,1.06,0.53,0.58
seasonal naive (L=168),1.00,1.00,1.00,1.00,1.00
Chronos-2 independent,1.75,1.68,1.15,0.96,0.80
Chronos-2 joint,1.34,1.78,1.11,0.99,1.11
LSTM,0.81,0.74,0.85,0.63,0.72


Best model per pollutant:
 valeur_NO2     persistence
valeur_CO             LSTM
valeur_O3             LSTM
valeur_PM10    persistence
valeur_PM25    persistence
Best model on the average MAE: LSTM


In [70]:
# MAE relative to the pollutant's mean level, so that all five pollutants are comparable
level = train_imp[pollutants].mean()
rel = (results_all[pollutants] / level * 100).round(1)
fig = px.bar(rel.reset_index(names="model").melt(id_vars="model", var_name="pollutant", value_name="MAE (% of mean level)"),
             x="pollutant", y="MAE (% of mean level)", color="model", barmode="group",
             title="Validation MAE per pollutant<br><sup>as a percentage of the pollutant's mean level</sup>")
fig.update_layout(margin=dict(t=100))
fig.show()

Analyse whether performance changes between the beginning and the end of the 504-hour horizon.

In [71]:
# MAE per week of the horizon (week 1 = hours 1-168, week 2 = 169-336, week 3 = 337-504), as % of the mean level
week = pd.Series(np.arange(HOURS) // 168 + 1, index=val.index, name="week")
rows = []
for name, pred in preds.items():
    err = (val[pollutants] - pred[pollutants]).abs().groupby(week).mean() / level * 100
    for w in err.index:
        rows.append({"model": name, "week": f"week {w}", **err.loc[w].to_dict(), "average": err.loc[w].mean()})
by_week = pd.DataFrame(rows)
display(by_week.round(1).set_index(["model", "week"]))

fig = px.line(by_week, x="week", y="average", color="model", markers=True,
              title="MAE by week of the horizon <br><sup>(average over pollutants, % of mean level)</sup>",
              labels={"average": "MAE (% of mean level)"})
fig.show()

valeur_NO2  valeur_CO  valeur_O3  valeur_PM10  \
model                  week                                                    
persistence            week 1        44.7       23.4       32.8         23.4   
                       week 2        35.3       19.5       31.6         29.6   
                       week 3        25.3       20.7       45.7         25.4   
seasonal naive (L=168) week 1        48.1       22.9       32.9         46.3   
                       week 2        43.5       21.3       29.9         50.9   
                       week 3        54.5       23.0       41.2         51.1   
Chronos-2 independent  week 1        54.4       33.4       32.4         39.5   
                       week 2       113.6       38.7       38.1         52.8   
                       week 3        87.8       40.8       49.1         49.6   
Chronos-2 joint        week 1        52.5       36.7       33.4         41.2   
                       week 2        62.8       37.1       33.1         50.3   
                       week 3        81.2       45.9       48.4         55.2   
LSTM                   week 1        40.8       19.4       28.7         24.5   
                       week 2        34.7       14.7       25.5         33.4   
                       week 3        42.7       15.7       34.6         35.0   

                               valeur_PM25  average  
model                  week                          
persistence            week 1         21.3     29.1  
                       week 2         29.1     29.0  
                       week 3         26.0     28.6  
seasonal naive (L=168) week 1         44.9     39.0  
                       week 2         44.2     38.0  
                       week 3         42.6     42.5  
Chronos-2 independent  week 1         27.6     37.4  
                       week 2         38.4     56.3  
                       week 3         40.0     53.5  
Chronos-2 joint        week 1         37.0     40.2  
                       week 2         48.6     46.4  
                       week 3         60.2     58.2  
LSTM                   week 1         23.0     27.3  
                       week 2         34.0     28.4  
                       week 3         37.5     33.1

Plot observed vs predicted values.

In [29]:
best_name = results_all["average"].idxmin()
fig = make_subplots(rows=len(pollutants), cols=1, shared_xaxes=True, subplot_titles=pollutants)
for i, col in enumerate(pollutants, start=1):
    for name, series, color in [("observed", val[col], "black"),
                                ("seasonal naive (L=168)", pred_snaive[col], "blue"),
                                ("Chronos-2 joint", pred_chronos_joint[col], "green"),
                                (f"best on average: {best_name}", preds[best_name][col], "red")]:
        fig.add_trace(go.Scatter(x=val.index, y=series, name=name, line=dict(color=color), legendgroup=name, showlegend=(i == 1)), row=i, col=1)
fig.update_layout(title="Observed vs predicted, validation period", height=1200)
fig.show()

In [72]:
# Where does Chronos-2 beat or lose to seasonal naive most clearly? (daily MAE gap per pollutant, % of mean level)
daily_gap = ((val[pollutants] - pred_chronos_joint[pollutants]).abs() - (val[pollutants] - pred_snaive[pollutants]).abs()).resample("D").mean() / level * 100
fig = px.imshow(daily_gap.T, color_continuous_scale="RdBu_r", color_continuous_midpoint=0, aspect="auto",
                title="Daily MAE of Chronos-2 joint minus seasonal naive <br><sup>(% of mean level; negative = Chronos better)</sup>")
fig.show()

**Comment on one clear failure: Chronos-2 on NO2.** The zero-shot forecast reproduces the daily shape well (morning and evening rush-hour peaks), but its level is wrong. The model mean-reverts to the level of its recent context: the two weeks before the validation period were polluted (mean NO2 about 39 ug/m3, with an episode around 4-10 October), so it forecasts a mean of about 32 ug/m3, whereas the validation period was much cleaner (about 18 ug/m3). The level error repeats every day, which gives an MAE of 14.8 against 7.9 for persistence, whose last observed value happened to be low. The same level shift explains the weaker results for CO and PM10, and the gap grows in weeks 2 and 3.

### Part Six: Final forecast and submission

Select the final method using validation results only, then generate forecasts for the competition test period.

In [31]:
forecasters = {
    "persistence": persistence,
    "seasonal naive (L=168)": lambda h, idx: seasonal_naive(h, idx, L=168),
    "Chronos-2 independent": lambda h, idx: chronos_forecast(h, idx, joint=False),
    "Chronos-2 joint": lambda h, idx: chronos_forecast(h, idx, joint=True),
    "LSTM": lambda h, idx: forecast_lstm(fit_lstm(h), h, idx),
}

# 1. selection: lowest average validation MAE
final_name = results_all["average"].idxmin()
print("Selected method:", final_name)

# 2. history for the test forecast = the whole training file, cleaned and imputed exactly like the training part above
full = df_train.copy()
full_imp = full.copy()
for col in pollutants:
    prof = full[col].groupby([full.index.dayofweek, full.index.hour]).mean()
    full_imp[col], _ = impute_past_only(full[col], prof)

# 3. test timestamps: the 504 hours that follow the last training hour
test_index = pd.DatetimeIndex(pd.to_datetime(df_test["id"], format="%Y-%m-%d %H"))
assert test_index[0] == full_imp.index[-1] + pd.Timedelta(hours=1)

test_pred = forecasters[final_name](full_imp, test_index)
test_pred.round(3).head()

Selected method: LSTM


epoch 5: train L1 (standardised) = 0.6101


epoch 10: train L1 (standardised) = 0.5884


epoch 15: train L1 (standardised) = 0.5813


epoch 20: train L1 (standardised) = 0.6130


epoch 25: train L1 (standardised) = 0.5899


,valeur_NO2,valeur_CO,valeur_O3,valeur_PM10,valeur_PM25
id,,,,,
2024-09-03 23:00:00,19.110,0.189,45.131,14.439,5.829
2024-09-04 00:00:00,17.315,0.193,44.069,14.913,6.287
2024-09-04 01:00:00,15.541,0.173,44.300,14.424,6.083
2024-09-04 02:00:00,15.293,0.171,41.719,12.967,5.238
2024-09-04 03:00:00,19.223,0.169,38.197,13.314,5.477


Check that the final submission follows the required format and contains complete, correctly ordered predictions for all five pollutants.

In [32]:
submission = test_pred[pollutants].reset_index(drop=True)
submission.insert(0, "id", df_test["id"].values)       # keep the id strings exactly as in test.csv

assert list(submission.columns) == ["id"] + list(df_train.columns[:5]), "column names / order differ from train.csv"
assert len(submission) == len(df_test) == HOURS
assert (submission["id"].values == df_test["id"].values).all(), "ids are not in test.csv order"
assert pd.to_datetime(submission["id"], format="%Y-%m-%d %H").is_monotonic_increasing
assert submission[pollutants].notna().all().all(), "missing predictions"
assert np.isfinite(submission[pollutants].values).all()
assert (submission[pollutants] >= 0).all().all(), "negative concentrations"

submission.to_csv("../submission.csv", index=False)
print(submission.shape); display(submission.head()); display(submission.describe().round(2))

(504, 6)


,id,valeur_NO2,valeur_CO,valeur_O3,valeur_PM10,valeur_PM25
0,2024-09-03 23,19.110351,0.189411,45.131242,14.438556,5.828888
1,2024-09-04 00,17.315390,0.193489,44.069336,14.912545,6.286756
2,2024-09-04 01,15.541396,0.172773,44.299925,14.423542,6.082757
3,2024-09-04 02,15.293034,0.170601,41.719478,12.966569,5.238186
4,2024-09-04 03,19.222880,0.169338,38.197362,13.314082,5.477401


,valeur_NO2,valeur_CO,valeur_O3,valeur_PM10,valeur_PM25
count,504.00,504.00,504.00,504.00,504.00
mean,16.65,0.18,58.27,16.98,8.29
std,4.09,0.01,10.95,2.68,0.98
min,8.44,0.15,29.97,10.15,5.24
25%,13.79,0.17,50.70,15.25,7.58
50%,16.34,0.18,59.51,17.13,8.28
75%,19.22,0.19,67.33,18.81,8.98
max,29.09,0.22,76.79,22.86,11.25


In [33]:
# sanity plot: last 4 weeks of history followed by the forecast
hist = full_imp[pollutants].iloc[-24 * 28:]
fig = make_subplots(rows=len(pollutants), cols=1, shared_xaxes=True, subplot_titles=pollutants)
for i, col in enumerate(pollutants, start=1):
    fig.add_trace(go.Scatter(x=hist.index, y=hist[col], name="history", line=dict(color="black"), legendgroup="h", showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=test_index, y=test_pred[col], name=f"forecast ({final_name})", line=dict(color="red"), legendgroup="f", showlegend=(i == 1)), row=i, col=1)
fig.update_layout(title="Final test forecast", height=1100)
fig.show()